In [1]:
import pandas as pd
import seaborn as sns
df = sns.load_dataset('titanic')
emb = df['embarked'].fillna('S') # Zer egiten du kode honek? -> embarked-en 2 nuluak 'S'-rekin (moda, portu ohikoena) betetzen ditu
print(emb.value_counts())

embarked
S    646
C    168
Q     77
Name: count, dtype: int64


In [2]:
# Zer egiten du hurrengo kodeak?
# LabelEncoder-ek kategoria bakoitzari zenbaki oso bat ematen dio ORDEN ALFABETIKOAN: C=0, Q=1, S=2.
# fit -> kategoriak ikasi (le.classes_), transform -> zenbaki bihurtu.
# Arazoa: 0<1<2 ordena faltsu bat sartzen du (portuek ez dute ordenarik) -> nominaletan hobe one-hot.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
emb_label = le.fit_transform(emb)
print('Kategoriak:', list(le.classes_))
print('Kodetua (lehen 8):', emb_label[:8])

Kategoriak: ['C', 'Q', 'S']
Kodetua (lehen 8): [2 0 2 2 2 1 2 2]


In [3]:
# One-hot: kategoria bakoitzeko 0/1 zutabe bat; drop_first=True -> lehenengoa (emb_C) kendu
emb_oh = pd.get_dummies(emb, prefix='emb', drop_first=True)
print('Zutabeak:', emb_oh.columns.tolist())
emb_oh.head()

Zutabeak: ['emb_Q', 'emb_S']


,emb_Q,emb_S
0,False,True
1,False,False
2,False,True
3,False,True
4,False,True


In [4]:
# Jatorrizkoa, label eta one-hot elkarren ondoan konparatzeko
konparaketa = pd.DataFrame({'jatorrizkoa': emb, 'label':
emb_label}).join(emb_oh)
konparaketa.head(8)

,jatorrizkoa,label,emb_Q,emb_S
0,S,2,False,True
1,C,0,False,False
2,S,2,False,True
3,S,2,False,True
4,S,2,False,True
5,Q,1,True,False
6,S,2,False,True
7,S,2,False,True


**Aldagai kategorikoak eta beren kategoriak:**

- `sex` : 2 kategoria → `['male', 'female']`
- `embarked` : 3 kategoria → `['S', 'C', 'Q']`
- `class` : 3 kategoria → `['First', 'Second', 'Third']`

**Embarked kasuan 3 kategoria baditugu (C, Q, S), zergatik uzten ditugu 2 zutabe eta ez 3? Zer du txarra 3 zutabe uzteak?**

Hirugarren zutabea **deduzitu** egin daitekeelako: `drop_first=True`-k `emb_C` kentzen du, eta `emb_Q = 0` eta `emb_S = 0` badira, bidaiaria derrigorrez C-n igo zen. Informazio bera 2 zutaberekin dago.

3 zutabe uzteak **erredundantzia** sortzen du: hiru zutabeen batura beti 1 da (multikolinealtasun perfektua, *dummy variable trap*). Modelo linealetan (erregresio lineala/logistikoa) horrek koefizienteak ezegonkor eta interpretaezin bihurtzen ditu. Zuhaitzetan ez da hain larria, baina zutabe alferrikakoa da.

**Zein kodearekin kentzen da zutabe bat?**

`pd.get_dummies(..., drop_first=True)` — lehenengo kategoria (orden alfabetikoan, hemen `C`) kentzen du.